In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
PROJECT_ROOT = Path("/home/gdanzil/econ470/a0/work/hcris-data")

In [4]:
# Variable locations ------------------------------------------------------

hcris_vars = pd.DataFrame([
    ['beds', 'S300001', '01400', '00200', 'numeric'],
    ['tot_charges', 'G300000', '00100', '00100', 'numeric'],
    ['tot_discounts', 'G300000', '00200', '00100', 'numeric'],
    ['net_pat_rev', 'G300000', '00300', '00100', 'numeric'],
    ['tot_operating_exp', 'G300000', '00400', '00100', 'numeric'],
    ['ip_charges', 'G200000', '00100', '00100', 'numeric'],
    ['icu_charges', 'G200000', '01600', '00100', 'numeric'],
    ['ancillary_charges', 'G200000', '01800', '00100', 'numeric'],
    ['tot_discharges', 'S300001', '00100', '01500', 'numeric'],
    ['mcare_discharges', 'S300001', '00100', '01300', 'numeric'],
    ['mcaid_discharges', 'S300001', '00100', '01400', 'numeric'],
    ['tot_mcare_payment', 'E00A18A', '05900', '00100', 'numeric'],
    ['secondary_mcare_payment', 'E00A18A', '06000', '00100', 'numeric'],
    ['street', 'S200001', '00100', '00100', 'alpha'],
    ['city', 'S200001', '00200', '00100', 'alpha'],
    ['state', 'S200001', '00200', '00200', 'alpha'],
    ['zip', 'S200001', '00200', '00300', 'alpha'],
    ['county', 'S200001', '00200', '00400', 'alpha'],
    ['name', 'S200001', '00300', '00100', 'alpha'],
    ['hvbp_payment', 'E00A18A', '07093', '00100', 'numeric'],
    ['hrrp_payment', 'E00A18A', '07094', '00100', 'numeric'],
    ['tot_uncomp_care_charges', 'S100000', '02000', '00300', 'numeric'],
    ['tot_uncomp_care_partial_pmts', 'S100000', '02200', '00300', 'numeric'],
    ['bad_debt', 'S100000', '02800', '00100', 'numeric'],
    ['cost_to_charge', 'S100000', '00100', '00100', 'numeric'],
    ['new_cap_ass', 'A700001', '01000', '00200', 'numeric'],
    ['cash', 'G000000', '00100', '00100', 'numeric'],
    ['fixed_assets', 'G000000', '03000', '00100', 'numeric'],
    ['depr_land', 'G000000', '01400', '00100', 'numeric'],
    ['depr_bldg', 'G000000', '01600', '00100', 'numeric'],
    ['depr_lease', 'G000000', '01800', '00100', 'numeric'],
    ['depr_fixed_equip', 'G000000', '02000', '00100', 'numeric'],
    ['depr_auto', 'G000000', '02200', '00100', 'numeric'],
    ['depr_major_equip', 'G000000', '02400', '00100', 'numeric'],
    ['depr_minor_equip', 'G000000', '02600', '00100', 'numeric'],
    ['depr_HIT', 'G000000', '02800', '00100', 'numeric'],
    ['current_assets', 'G000000', '01100', '00100', 'numeric'],
    ['current_liabilities', 'G000000', '04500', '00100', 'numeric'],
], columns=["variable", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "source"])


# Import data -------------------------------------------------------------

def hcris_v2010_path(year: int, suffix: str):
    """
    Find a file either in an unzipped folder or inside a ZIP archive.
    Returns a dict like:
      {"type": "file", "path": Path(...)}
      {"type": "zip", "path": Path(...), "entry": "..."}
    """
    base = Path("../hcris-data/HCRIS_v2010")

    # Try unzipped folders first
    folders = [f"HospitalFY{year}", f"HOSP10FY{year}"]
    files = [
        f"hosp10_{year}_{suffix.upper()}.CSV",
        f"HOSP10_{year}_{suffix.lower()}.csv",
    ]

    for folder in folders:
        for file in files:
            path = base / folder / file
            if path.exists():
                return {"type": "file", "path": path}

    # Fall back to ZIP archive
    zips = [
        f"HOSP10FY{year}.ZIP",
        f"HOSP10FY{year}.zip",
        f"HospitalFY{year}.ZIP",
        f"HospitalFY{year}.zip",
    ]
    zip_entries = [
        f"HOSP10_{year}_{suffix.lower()}.csv",
        f"hosp10_{year}_{suffix.upper()}.CSV",
    ]

    for zf in zips:
        zip_path = base / zf
        if zip_path.exists():
            try:
                with ZipFile(zip_path, "r") as z:
                    contents = set(z.namelist())
                    for entry in zip_entries:
                        if entry in contents:
                            return {"type": "zip", "path": zip_path, "entry": entry}
            except Exception:
                pass

    raise FileNotFoundError(f"Cannot find HCRIS v2010 data for year {year}, suffix {suffix}")


def hcris_v2010_read(year: int, suffix: str, col_names: list[str]) -> pd.DataFrame:
    """
    Read either from a normal CSV or from a CSV stored inside a ZIP.
    """
    loc = hcris_v2010_path(year, suffix)

    if loc["type"] == "file":
        return pd.read_csv(
            loc["path"],
            header=None,
            names=col_names,
            dtype=str,
            low_memory=False
        )

    with ZipFile(loc["path"], "r") as z:
        with z.open(loc["entry"]) as f:
            return pd.read_csv(
                f,
                header=None,
                names=col_names,
                dtype=str,
                low_memory=False
            )


all_years = []

for year in range(2010, 2019):
    hcris_alpha = hcris_v2010_read(
        year,
        "alpha",
        col_names=['RPT_REC_NUM', 'WKSHT_CD', 'LINE_NUM', 'CLMN_NUM', 'ITM_VAL_NUM']
    )

    hcris_numeric = hcris_v2010_read(
        year,
        "nmrc",
        col_names=['RPT_REC_NUM', 'WKSHT_CD', 'LINE_NUM', 'CLMN_NUM', 'ITM_VAL_NUM']
    )

    hcris_report = hcris_v2010_read(
        year,
        "rpt",
        col_names=[
            'RPT_REC_NUM', 'PRVDR_CTRL_TYPE_CD', 'PRVDR_NUM', 'NPI',
            'RPT_STUS_CD', 'FY_BGN_DT', 'FY_END_DT', 'PROC_DT',
            'INITL_RPT_SW', 'LAST_RPT_SW', 'TRNSMTL_NUM', 'FI_NUM',
            'ADR_VNDR_CD', 'FI_CREAT_DT', 'UTIL_CD', 'NPR_DT',
            'SPEC_IND', 'FI_RCPT_DT'
        ]
    )

    final_reports = hcris_report[[
        'RPT_REC_NUM', 'PRVDR_NUM', 'NPI',
        'FY_BGN_DT', 'FY_END_DT', 'PROC_DT',
        'FI_CREAT_DT', 'RPT_STUS_CD'
    ]].rename(columns={
        'RPT_REC_NUM': 'report',
        'PRVDR_NUM': 'provider_number',
        'NPI': 'npi',
        'FY_BGN_DT': 'fy_start',
        'FY_END_DT': 'fy_end',
        'PROC_DT': 'date_processed',
        'FI_CREAT_DT': 'date_created',
        'RPT_STUS_CD': 'status'
    })

    final_reports['year'] = year
    final_reports['data_source'] = 'v2010'

    for _, row in hcris_vars.iterrows():
        source_df = hcris_alpha if row['source'] == 'alpha' else hcris_numeric

        val = source_df[
            (source_df['WKSHT_CD'] == row['WKSHT_CD']) &
            (source_df['LINE_NUM'] == row['LINE_NUM']) &
            (source_df['CLMN_NUM'] == row['CLMN_NUM'])
        ][['RPT_REC_NUM', 'ITM_VAL_NUM']].rename(columns={
            'RPT_REC_NUM': 'report',
            'ITM_VAL_NUM': row['variable']
        })

        final_reports = final_reports.merge(val, on='report', how='left')

    all_years.append(final_reports)

final_hcris_v2010 = pd.concat(all_years, ignore_index=True)

output_path = Path("data/output")
output_path.mkdir(parents=True, exist_ok=True)

final_hcris_v2010.to_csv(
    output_path / "HCRIS_Data_v2010.txt",
    sep="\t",
    index=False
)